# 03 — Lip-Sync Server (Wav2Lip)

שרת FastAPI שלוקח **תמונת פנים + קובץ אודיו** ומחזיר **וידאו talking-head 9:16**.

- מודל: `wav2lip_gan.pth` (GAN, איכות שפתיים טובה יותר)
- דורש GPU (T4 מספיק)
- Post-processing: ffmpeg ל-crop/pad 1080×1920 + חידוד קל

**לפני הרצה:** Runtime → T4 GPU.

## CELL 1 — התקנת תלויות בסיסיות

FastAPI + HTTP client + OpenCV + ffmpeg. החבילות של Wav2Lip עצמו יותקנו בתא הבא.

In [ ]:
# חבילות ליבה לשרת + ffmpeg לעיבוד וידאו
!pip install -q fastapi uvicorn nest-asyncio httpx aiofiles
!pip install -q opencv-python-headless ffmpeg-python
!apt-get install -y -qq ffmpeg

## CELL 2 — Clone + setup של Wav2Lip

שלבים:
1. שכפול ה-repo הרשמי
2. התקנת requirements של Wav2Lip (עם patch ל-librosa/numpy שעובד ב-Colab המודרני)
3. הורדת המשקולות `wav2lip_gan.pth`
4. הורדת מודל ה-face-detection (s3fd) שדרוש לסינון פריימים

**הערה:** קישור ה-SharePoint של המשקולות המקורי לפעמים נופל — ולכן הוספתי מראה ב-HuggingFace כ-fallback.

In [ ]:
import os

# 1. שכפול
if not os.path.isdir('/content/Wav2Lip'):
    !git clone -q https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
%cd /content/Wav2Lip

# 2. התקנת requirements של Wav2Lip
#    הגרסאות בקובץ המקורי (librosa==0.7.0) לא תואמות Python 3.11/3.12.
#    מתקינים גרסאות תואמות Colab עכשווי.
!pip install -q librosa==0.10.2 numpy==1.26.4
!pip install -q numba==0.60.0 resampy==0.4.3
!pip install -q batch-face
!pip install -q -r requirements.txt || echo '(some optional reqs may fail, continuing)'

# 3. הורדת wav2lip_gan.pth — SharePoint קודם, HuggingFace כ-fallback
os.makedirs('checkpoints', exist_ok=True)
CKPT = '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'

if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 100_000_000:
    print('Downloading wav2lip_gan.pth (~416MB)...')
    SHAREPOINT = (
        'https://iiitaphyd-my.sharepoint.com/personal/radrabha_m_research_iiit_ac_in/'
        '_layouts/15/download.aspx?share=EdjI7bZlgApMqrnYJVFHTaYBo4b0HmAUJO_hqJNUVIJlHA'
    )
    HF_MIRROR = (
        'https://huggingface.co/camenduru/Wav2Lip/resolve/main/wav2lip_gan.pth'
    )
    # מנסים SharePoint; אם הקובץ יוצא קטן מדי — עוברים למראה
    !wget -q --show-progress "{SHAREPOINT}" -O {CKPT} || true
    if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 100_000_000:
        print('SharePoint link failed — trying HuggingFace mirror')
        !wget -q --show-progress "{HF_MIRROR}" -O {CKPT}
    print(f'checkpoint size: {os.path.getsize(CKPT)/1e6:.1f} MB')

# 4. הורדת s3fd (face detector) — Wav2Lip מחפש אותו ב-face_detection/detection/sfd/s3fd.pth
S3FD = '/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth'
if not os.path.exists(S3FD):
    print('Downloading s3fd face detector...')
    !wget -q 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth' -O {S3FD} \
      || wget -q 'https://huggingface.co/camenduru/Wav2Lip/resolve/main/s3fd.pth' -O {S3FD}
    print(f's3fd size: {os.path.getsize(S3FD)/1e6:.1f} MB')

print('Wav2Lip setup complete.')

## CELL 3 — חיבור Drive ותיקיות פלט

פלט ה-lip-sync נשמר ב-`outputs/lipsync/{job_id}.mp4`. קבצי הקלט הזמניים נשארים ב-`/tmp` ונמחקים ע"י ה-runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/viral_empire'
LIPSYNC_OUT = f'{BASE}/outputs/lipsync'
os.makedirs(LIPSYNC_OUT, exist_ok=True)
os.makedirs(f'{BASE}/outputs/images', exist_ok=True)
os.makedirs(f'{BASE}/outputs/tts', exist_ok=True)

print(f'LIPSYNC_OUT = {LIPSYNC_OUT}')

## CELL 4 — FastAPI server + בדיקת דגימה

### Endpoints
- `POST /lipsync` — מקבל face + audio (URL או נתיב Drive), מחזיר MP4 9:16
- `GET  /health`
- `GET  /jobs/{job_id}` — בודק אם הקובץ מוכן
- `GET  /jobs/{job_id}/download` — הורדת הוידאו הסופי

### Pipeline
1. הורדה מקומית של `face` + `audio` ל-`/tmp`
2. `python inference.py --checkpoint_path ... --resize_factor 1 --nosmooth`
3. ffmpeg: crop/pad ל-1080×1920 + unsharp חידוד קל + H.264 + AAC
4. callback אופציונלי ל-backend המקומי

בסוף התא — בדיקה שמריצה pipeline אמיתי אם נמצאים דגימות קלט.

In [ ]:
# ---------- שרת FastAPI עבור lip-sync ----------
import asyncio
import glob
import os
import shutil
import socket
import subprocess
import threading
import time
import traceback
from typing import Optional

import httpx
import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field

nest_asyncio.apply()

WAV2LIP_DIR = '/content/Wav2Lip'
CHECKPOINT = f'{WAV2LIP_DIR}/checkpoints/wav2lip_gan.pth'
GPU_AVAILABLE = torch.cuda.is_available()

app = FastAPI(title='Viral Empire — Lip-Sync Server (Wav2Lip)', version='0.1.0')

class LipSyncRequest(BaseModel):
    job_id: str = Field(..., min_length=1)
    face_image_url: str = Field(..., description='HTTP URL or local/Drive path to face image')
    audio_url: str = Field(..., description='HTTP URL or local/Drive path to audio file')
    callback_url: Optional[str] = None

# ---------- helpers ----------
async def _fetch_to_file(src: str, dst: str) -> str:
    """מוריד URL או מעתיק נתיב מקומי/Drive לקובץ יעד."""
    if src.startswith(('http://', 'https://')):
        async with httpx.AsyncClient(timeout=120, follow_redirects=True) as http:
            async with http.stream('GET', src) as resp:
                resp.raise_for_status()
                with open(dst, 'wb') as f:
                    async for chunk in resp.aiter_bytes(chunk_size=65536):
                        f.write(chunk)
    else:
        if not os.path.exists(src):
            raise FileNotFoundError(f'not found: {src}')
        shutil.copyfile(src, dst)
    return dst

def _run_wav2lip(face_path: str, audio_path: str, out_path: str) -> None:
    """קורא ל-inference.py של Wav2Lip עם retry ב-OOM."""
    cmd = [
        'python', 'inference.py',
        '--checkpoint_path', CHECKPOINT,
        '--face', face_path,
        '--audio', audio_path,
        '--outfile', out_path,
        '--resize_factor', '1',
        '--nosmooth',
    ]
    result = subprocess.run(
        cmd, cwd=WAV2LIP_DIR, capture_output=True, text=True, timeout=900
    )
    # Wav2Lip זורק CUDA OOM כ-RuntimeError; מנקים cache ומנסים שוב פעם אחת
    combined = (result.stdout or '') + '\n' + (result.stderr or '')
    if result.returncode != 0 and 'out of memory' in combined.lower():
        print('[wav2lip] GPU OOM — clearing cache and retrying with batch 8...')
        torch.cuda.empty_cache()
        cmd_retry = cmd + ['--wav2lip_batch_size', '8', '--face_det_batch_size', '4']
        result = subprocess.run(
            cmd_retry, cwd=WAV2LIP_DIR, capture_output=True, text=True, timeout=900
        )
    if result.returncode != 0:
        raise RuntimeError(
            f'wav2lip failed (rc={result.returncode}).\n'
            f'stdout: {result.stdout[-800:]}\n'
            f'stderr: {result.stderr[-800:]}'
        )

def _post_process_9x16(src_mp4: str, dst_mp4: str) -> None:
    """Crop/pad ל-1080x1920 + unsharp קל + H.264 + AAC."""
    # scale כך שהגובה הוא 1920 (או הרוחב 1080), ואז pad שחור סביב + unsharp
    vf = (
        "scale='if(gt(a,9/16),1080,-2)':'if(gt(a,9/16),-2,1920)',"
        'pad=1080:1920:(ow-iw)/2:(oh-ih)/2:color=black,'
        'unsharp=5:5:0.8:3:3:0.4'
    )
    cmd = [
        'ffmpeg', '-y', '-i', src_mp4,
        '-vf', vf,
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '20',
        '-pix_fmt', 'yuv420p',
        '-c:a', 'aac', '-b:a', '192k',
        '-movflags', '+faststart',
        dst_mp4,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        raise RuntimeError(f'ffmpeg failed: {result.stderr[-800:]}')

# ---------- /lipsync ----------
@app.post('/lipsync')
async def lipsync(req: LipSyncRequest):
    face_path = f'/tmp/{req.job_id}_face.jpg'
    audio_path = f'/tmp/{req.job_id}_audio.wav'
    raw_out = os.path.join(LIPSYNC_OUT, f'{req.job_id}_raw.mp4')
    final_out = os.path.join(LIPSYNC_OUT, f'{req.job_id}.mp4')

    # 1+2. הורדת קלטים
    try:
        await _fetch_to_file(req.face_image_url, face_path)
        await _fetch_to_file(req.audio_url, audio_path)
    except FileNotFoundError as exc:
        raise HTTPException(404, str(exc)) from exc
    except Exception as exc:
        raise HTTPException(400, f'fetch failed: {exc}') from exc

    # 3. Wav2Lip inference
    try:
        await asyncio.to_thread(_run_wav2lip, face_path, audio_path, raw_out)
    except Exception as exc:
        traceback.print_exc()
        raise HTTPException(500, f'lipsync failed: {exc}') from exc

    # 4. Post-process ל-9:16
    try:
        await asyncio.to_thread(_post_process_9x16, raw_out, final_out)
    except Exception as exc:
        traceback.print_exc()
        raise HTTPException(500, f'postprocess failed: {exc}') from exc

    result = {
        'job_id': req.job_id,
        'status': 'done',
        'path': final_out,
        'size_bytes': os.path.getsize(final_out),
        'download_url': f'/jobs/{req.job_id}/download',
    }

    # 5. callback אופציונלי
    if req.callback_url:
        try:
            async with httpx.AsyncClient(timeout=15) as http:
                await http.post(req.callback_url, json=result)
        except Exception as exc:
            print(f'callback failed: {exc}')
            result['callback_error'] = str(exc)

    return result

# ---------- /health ----------
@app.get('/health')
def health():
    return {
        'status': 'ok',
        'model': 'wav2lip_gan',
        'gpu': bool(torch.cuda.is_available()),
        'checkpoint_exists': os.path.exists(CHECKPOINT),
    }

# ---------- /jobs/{job_id} ----------
@app.get('/jobs/{job_id}')
def job_status(job_id: str):
    path = os.path.join(LIPSYNC_OUT, f'{job_id}.mp4')
    if not os.path.exists(path):
        return {'job_id': job_id, 'status': 'pending', 'exists': False}
    return {
        'job_id': job_id,
        'status': 'done',
        'exists': True,
        'size_bytes': os.path.getsize(path),
        'path': path,
    }

@app.get('/jobs/{job_id}/download')
def job_download(job_id: str):
    path = os.path.join(LIPSYNC_OUT, f'{job_id}.mp4')
    if not os.path.exists(path):
        raise HTTPException(404, 'job not ready')
    return FileResponse(path, media_type='video/mp4', filename=f'{job_id}.mp4')

# ---------- הרצה ב-thread רקע ----------
PORT = 8000

def _run_server():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

for _ in range(30):
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            break
    except OSError:
        time.sleep(1)
print(f'FastAPI running on http://127.0.0.1:{PORT}')

# =====================================================================
# בדיקת דגימה — מריצה pipeline אמיתי אם יש קלטים ב-Drive
#
# הבדיקה מחפשת:
#   - תמונת פנים כלשהי ב-outputs/images/*.jpg
#   - קובץ אודיו כלשהו ב-outputs/tts/*.wav
# אם לא נמצאו — מדלגת ומדפיסה הוראות איך לבחון ידנית.
# =====================================================================
sample_faces = sorted(glob.glob(f'{BASE}/outputs/images/*.jpg'))
sample_audios = sorted(glob.glob(f'{BASE}/outputs/tts/*.wav'))

if sample_faces and sample_audios:
    print(f'\nFound sample inputs — running pipeline test')
    print(f'  face:  {sample_faces[0]}')
    print(f'  audio: {sample_audios[0]}')
    async def _run_sample():
        return await lipsync(LipSyncRequest(
            job_id='_sample_lipsync',
            face_image_url=sample_faces[0],
            audio_url=sample_audios[0],
        ))
    try:
        sample_result = asyncio.get_event_loop().run_until_complete(_run_sample())
        print(f'Sample OK: {sample_result}')
    except Exception as exc:
        print(f'Sample failed: {exc}')
else:
    print('\n(skipping sample test — no face/audio found)')
    print(f'  expected face  in: {BASE}/outputs/images/*.jpg')
    print(f'  expected audio in: {BASE}/outputs/tts/*.wav')
    print('  (run the TTS + Image notebooks first to populate these)')

## CELL 5 — חשיפה דרך Cloudflare Tunnel

אותה טכניקה כמו ב-notebook TTS וה-notebook Image. שומר את ה-URL ל-`lipsync_server_url.txt` ב-Drive.

In [ ]:
# התקנת cloudflared והקמת tunnel
import os
import re
import subprocess
import time

if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Installing cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print('cloudflared installed')

LOG_PATH = '/content/cloudflared_lipsync.log'
if os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)

proc = subprocess.Popen(
    [
        'cloudflared', 'tunnel', '--no-autoupdate',
        '--url', f'http://localhost:{PORT}',
        '--logfile', LOG_PATH,
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f'cloudflared started (pid={proc.pid}), waiting for public URL...')

public_url = None
url_pattern = re.compile(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com')
for _ in range(60):
    time.sleep(1)
    if not os.path.exists(LOG_PATH):
        continue
    with open(LOG_PATH, 'r') as f:
        log_text = f.read()
    m = url_pattern.search(log_text)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError(f'Could not find tunnel URL. Check {LOG_PATH}')

URL_FILE = '/content/drive/MyDrive/viral_empire/lipsync_server_url.txt'
with open(URL_FILE, 'w') as f:
    f.write(public_url + '\n')

print('=' * 60)
print(f'PUBLIC URL: {public_url}')
print(f'Saved to:   {URL_FILE}')
print('=' * 60)
print(f'Test it:    curl {public_url}/health')

## CELL 6 — לולאת Keep-Alive

לא לסגור את הטאב — Colab מנתק runtime אחרי ~90 דקות חוסר פעילות.

In [ ]:
# Keep-alive — שמירה על ה-runtime
import time

print('Lip-sync server running. Keep this tab open.')
print(f'Public URL: {public_url}')

try:
    while True:
        time.sleep(30)
        print(f"Alive: {time.strftime('%H:%M:%S')}")
except KeyboardInterrupt:
    print('Stopped by user')